# Statistical Analysis

This section evaluates the relationships and differences between key sales, profit, discount, shipping, customer, and product variables.

### Tests Performed

* **Correlation Analysis**

  * Discount vs Profit
  * Discount vs Total Sales
  * Shipping Cost vs Profit
  * Shipping Cost vs Total Sales
  * Quantity vs Profit
  * Quantity vs Total Sales
  * Total Sales vs Profit

  **Test selection:** Pearson or Spearman correlation based on normality.

* **Group Comparison**

  * Market vs Profit / Total Sales
  * Region vs Profit / Total Sales
  * Segment vs Profit / Total Sales
  * Category vs Profit / Total Sales

  **Test selection:** One-Way ANOVA, Welch ANOVA, or Kruskal-Wallis based on normality and variance assumptions.

* **Categorical Association**

  * Category vs Returned

  **Test:** Chi-Square Test of Independence.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

In [3]:
fact = pd.read_excel("Data/FactSales.xlsx")
product = pd.read_excel("Data/DimProduct.xlsx")
customer = pd.read_excel("Data/DimCustomer.xlsx")
shipping = pd.read_excel("Data/DimShipping.xlsx")

print("FactSales:", fact.shape)
print("DimProduct:", product.shape)
print("DimCustomer:", customer.shape)
print("DimShipping:", shipping.shape)

FactSales: (49670, 18)
DimProduct: (10246, 4)
DimCustomer: (1589, 3)
DimShipping: (25033, 9)


In [4]:
df = fact.copy()

# Add Product information
df = df.merge(
    product[["Product ID", "Category"]],
    on="Product ID",
    how="left"
)

# Add Customer information
df = df.merge(
    customer[["Customer ID", "Segment"]],
    on="Customer ID",
    how="left"
)

# Add Shipping information
df = df.merge(
    shipping[["Shipping ID", "Ship Mode", "Region"]],
    on="Shipping ID",
    how="left"
)

print("Final dataset shape:", df.shape)

df.head()

Final dataset shape: (49670, 22)


,Row ID,Order ID,Product ID,Sales,Quantity,Discount,Profit,Shipping Cost,Customer ID,Order Priority,...,Returned,Shipping ID,Discounted,Profitable,Total Sales,Order Priority_N,Category,Segment,Ship Mode,Region
0,1,MX-2014-143658,OFF-LA-10002782,13.080000,3,0.0,4.560000,1.033,SC-20575,Medium,...,0,44634,0,1,39.240000,2,Office Supplies,Consumer,Standard Class,North
1,2,MX-2012-155047,FUR-FU-10004015,252.160004,8,0.0,90.720001,13.449,KW-16570,Medium,...,0,34096,0,1,2017.280029,2,Furniture,Consumer,Standard Class,South
2,3,MX-2012-155047,FUR-BO-10002352,193.279999,2,0.0,54.080002,9.627,KW-16570,Medium,...,0,34096,0,1,386.559998,2,Furniture,Consumer,Standard Class,South
3,4,MX-2012-155047,OFF-BI-10004428,35.439999,4,0.0,4.960000,1.371,KW-16570,Medium,...,0,34096,0,1,141.759995,2,Office Supplies,Consumer,Standard Class,South
4,5,MX-2012-155047,OFF-AR-10004594,71.599998,2,0.0,11.440000,3.787,KW-16570,Medium,...,0,34096,0,1,143.199997,2,Office Supplies,Consumer,Standard Class,South


In [5]:
required_columns = [
    "Discount",
    "Shipping Cost",
    "Quantity",
    "Profit",
    "Total Sales",
    "Returned",
    "Discounted",
    "Market",
    "Region",
    "Segment",
    "Category",
    "Ship Mode",
    "Order Priority_N"
]

print("Missing values:")
print(df[required_columns].isnull().sum())

print("\nUnique values:")
print(df[
    [
        "Market",
        "Region",
        "Segment",
        "Category",
        "Ship Mode",
        "Returned",
        "Discounted",
        "Order Priority_N"
    ]
].nunique())

Missing values:
Discount            0
Shipping Cost       0
Quantity            0
Profit              0
Total Sales         0
Returned            0
Discounted          0
Market              0
Region              0
Segment             0
Category            0
Ship Mode           0
Order Priority_N    0
dtype: int64

Unique values:
Market               7
Region              13
Segment              3
Category             3
Ship Mode            4
Returned             2
Discounted           2
Order Priority_N     4
dtype: int64


In [6]:
def normality_test(data):
    data = pd.Series(data).dropna()

    statistic, p_value = stats.normaltest(data)

    result = "Normal" if p_value >= 0.05 else "Not Normal"

    return statistic, p_value, result

In [7]:
def correlation_test(data, x_col, y_col):
    
    test_data = data[[x_col, y_col]].dropna()

    x_stat, x_p, x_normal = normality_test(test_data[x_col])
    y_stat, y_p, y_normal = normality_test(test_data[y_col])

    if x_normal == "Normal" and y_normal == "Normal":
        statistic, p_value = stats.pearsonr(
            test_data[x_col],
            test_data[y_col]
        )
        test_name = "Pearson Correlation"
    else:
        statistic, p_value = stats.spearmanr(
            test_data[x_col],
            test_data[y_col]
        )
        test_name = "Spearman Correlation"

    result = (
        "Significant relationship exists."
        if p_value < 0.05
        else
        "No significant relationship found."
    )

    print(f"\n{x_col} vs {y_col}")
    print()
    print(f"N = {len(test_data)}")
    print(f"{x_col} normality: {x_normal} (p = {x_p:.6e})")
    print(f"{y_col} normality: {y_normal} (p = {y_p:.6e})")
    print(f"Selected test: {test_name}")
    print(f"Correlation: {statistic:.4f}")
    print(f"P-value: {p_value:.6e}")

    if p_value < 0.05:
        print("Reject H0")
    else:
        print("Fail to Reject H0")

    print(f"Result: {result}")

    return {
        "Variable 1": x_col,
        "Variable 2": y_col,
        "Test": test_name,
        "Statistic": statistic,
        "P-value": p_value,
        "Result": result
    }

In [8]:
def group_comparison_test(data, group_col, value_col):
    
    test_data = data[[group_col, value_col]].dropna()

    groups = [
        group[value_col].values
        for _, group in test_data.groupby(group_col)
    ]

    group_names = test_data[group_col].unique()

    print(f"\n{group_col} vs {value_col}")
    print()

   
    normal_results = []

    for name, group in test_data.groupby(group_col):
        _, p_value, normality = normality_test(group[value_col])

        normal_results.append(normality)

        print(
            f"{name}: "
            f"Normality p-value = {p_value:.6e} : "
            f"{normality}"
        )

    levene_stat, levene_p = stats.levene(*groups)

    print(f"\nLevene test p-value: {levene_p:.6e}")

    all_normal = all(
        result == "Normal"
        for result in normal_results
    )

    equal_variance = levene_p >= 0.05


    if all_normal and equal_variance:

        statistic, p_value = stats.f_oneway(*groups)

        test_name = "One-Way ANOVA"

    elif all_normal and not equal_variance:

        statistic, p_value = stats.f_oneway(
            *groups,
            equal_var=False
        )

        test_name = "Welch ANOVA"

    else:

        statistic, p_value = stats.kruskal(*groups)

        test_name = "Kruskal-Wallis"

    print(f"\nSelected test: {test_name}")
    print(f"Statistic: {statistic:.4f}")
    print(f"P-value: {p_value:.6e}")

    if p_value < 0.05:
        print("Reject H0")
        print("Significant difference exists.")
    else:
        print("Fail to Reject H0")
        print("No significant difference found.")

    return {
        "Grouping Variable": group_col,
        "Value Variable": value_col,
        "Test": test_name,
        "Statistic": statistic,
        "P-value": p_value
    }

In [9]:
def chi_square_test(data, col1, col2):
    
    test_data = data[[col1, col2]].dropna()

    contingency_table = pd.crosstab(
        test_data[col1],
        test_data[col2]
    )

    statistic, p_value, dof, expected = stats.chi2_contingency(
        contingency_table
    )

    print(f"\n{col1} vs {col2}")
    print()

    print("\nContingency Table:")
    print(contingency_table)

    print(f"\nChi-Square Statistic: {statistic:.4f}")
    print(f"Degrees of Freedom: {dof}")
    print(f"P-value: {p_value:.6e}")

    if p_value < 0.05:
        print("Reject H0")
        print("Result: Significant association exists.")
    else:
        print("Fail to Reject H0")
        print("Result: No significant association found.")

    return {
        "Variable 1": col1,
        "Variable 2": col2,
        "Test": "Chi-Square",
        "Statistic": statistic,
        "P-value": p_value
    }

In [10]:
discount_profit = correlation_test(
    df,
    "Discount",
    "Profit"
)

discount_sales = correlation_test(
    df,
    "Discount",
    "Total Sales"
)


Discount vs Profit

N = 49670
Discount normality: Not Normal (p = 0.000000e+00)
Profit normality: Not Normal (p = 0.000000e+00)
Selected test: Spearman Correlation
Correlation: -0.5982
P-value: 0.000000e+00
Reject H0
Result: Significant relationship exists.

Discount vs Total Sales

N = 49670
Discount normality: Not Normal (p = 0.000000e+00)
Total Sales normality: Not Normal (p = 0.000000e+00)
Selected test: Spearman Correlation
Correlation: -0.2004
P-value: 0.000000e+00
Reject H0
Result: Significant relationship exists.


In [11]:
shipping_profit = correlation_test(
    df,
    "Shipping Cost",
    "Profit"
)

shipping_sales = correlation_test(
    df,
    "Shipping Cost",
    "Total Sales"
)


Shipping Cost vs Profit

N = 49670
Shipping Cost normality: Not Normal (p = 0.000000e+00)
Profit normality: Not Normal (p = 0.000000e+00)
Selected test: Spearman Correlation
Correlation: 0.4480
P-value: 0.000000e+00
Reject H0
Result: Significant relationship exists.

Shipping Cost vs Total Sales

N = 49670
Shipping Cost normality: Not Normal (p = 0.000000e+00)
Total Sales normality: Not Normal (p = 0.000000e+00)
Selected test: Spearman Correlation
Correlation: 0.8531
P-value: 0.000000e+00
Reject H0
Result: Significant relationship exists.


In [12]:
quantity_profit = correlation_test(
    df,
    "Quantity",
    "Profit"
)

quantity_sales = correlation_test(
    df,
    "Quantity",
    "Total Sales"
)


Quantity vs Profit

N = 49670
Quantity normality: Not Normal (p = 0.000000e+00)
Profit normality: Not Normal (p = 0.000000e+00)
Selected test: Spearman Correlation
Correlation: 0.1986
P-value: 0.000000e+00
Reject H0
Result: Significant relationship exists.

Quantity vs Total Sales

N = 49670
Quantity normality: Not Normal (p = 0.000000e+00)
Total Sales normality: Not Normal (p = 0.000000e+00)
Selected test: Spearman Correlation
Correlation: 0.6641
P-value: 0.000000e+00
Reject H0
Result: Significant relationship exists.


In [13]:
sales_profit = correlation_test(
    df,
    "Total Sales",
    "Profit"
)


Total Sales vs Profit

N = 49670
Total Sales normality: Not Normal (p = 0.000000e+00)
Profit normality: Not Normal (p = 0.000000e+00)
Selected test: Spearman Correlation
Correlation: 0.5448
P-value: 0.000000e+00
Reject H0
Result: Significant relationship exists.


In [14]:
market_profit = group_comparison_test(
    df,
    "Market",
    "Profit"
)

market_sales = group_comparison_test(
    df,
    "Market",
    "Total Sales"
)


Market vs Profit

APAC: Normality p-value = 0.000000e+00 : Not Normal
Africa: Normality p-value = 0.000000e+00 : Not Normal
Canada: Normality p-value = 8.140020e-104 : Not Normal
EMEA: Normality p-value = 0.000000e+00 : Not Normal
EU: Normality p-value = 0.000000e+00 : Not Normal
LATAM: Normality p-value = 0.000000e+00 : Not Normal
US: Normality p-value = 0.000000e+00 : Not Normal

Levene test p-value: 1.876851e-28

Selected test: Kruskal-Wallis
Statistic: 421.5830
P-value: 6.385676e-88
Reject H0
Significant difference exists.

Market vs Total Sales

APAC: Normality p-value = 0.000000e+00 : Not Normal
Africa: Normality p-value = 0.000000e+00 : Not Normal
Canada: Normality p-value = 4.289916e-120 : Not Normal
EMEA: Normality p-value = 0.000000e+00 : Not Normal
EU: Normality p-value = 0.000000e+00 : Not Normal
LATAM: Normality p-value = 0.000000e+00 : Not Normal
US: Normality p-value = 0.000000e+00 : Not Normal

Levene test p-value: 3.116185e-75

Selected test: Kruskal-Wallis
Statistic:

In [15]:
region_profit = group_comparison_test(
    df,
    "Region",
    "Profit"
)

region_sales = group_comparison_test(
    df,
    "Region",
    "Total Sales"
)


Region vs Profit

Africa: Normality p-value = 0.000000e+00 : Not Normal
Canada: Normality p-value = 8.140020e-104 : Not Normal
Caribbean: Normality p-value = 2.364647e-120 : Not Normal
Central: Normality p-value = 0.000000e+00 : Not Normal
Central Asia: Normality p-value = 2.443910e-168 : Not Normal
EMEA: Normality p-value = 0.000000e+00 : Not Normal
East: Normality p-value = 0.000000e+00 : Not Normal
North: Normality p-value = 0.000000e+00 : Not Normal
North Asia: Normality p-value = 0.000000e+00 : Not Normal
Oceania: Normality p-value = 0.000000e+00 : Not Normal
South: Normality p-value = 0.000000e+00 : Not Normal
Southeast Asia: Normality p-value = 1.899614e-222 : Not Normal
West: Normality p-value = 0.000000e+00 : Not Normal

Levene test p-value: 2.769758e-31

Selected test: Kruskal-Wallis
Statistic: 1740.4292
P-value: 0.000000e+00
Reject H0
Significant difference exists.

Region vs Total Sales

Africa: Normality p-value = 0.000000e+00 : Not Normal
Canada: Normality p-value = 4.28

In [16]:
segment_profit = group_comparison_test(
    df,
    "Segment",
    "Profit"
)

segment_sales = group_comparison_test(
    df,
    "Segment",
    "Total Sales"
)


Segment vs Profit

Consumer: Normality p-value = 0.000000e+00 : Not Normal
Corporate: Normality p-value = 0.000000e+00 : Not Normal
Home Office: Normality p-value = 0.000000e+00 : Not Normal

Levene test p-value: 6.451430e-01

Selected test: Kruskal-Wallis
Statistic: 0.6225
P-value: 7.325238e-01
Fail to Reject H0
No significant difference found.

Segment vs Total Sales

Consumer: Normality p-value = 0.000000e+00 : Not Normal
Corporate: Normality p-value = 0.000000e+00 : Not Normal
Home Office: Normality p-value = 0.000000e+00 : Not Normal

Levene test p-value: 8.995258e-01

Selected test: Kruskal-Wallis
Statistic: 0.4421
P-value: 8.016610e-01
Fail to Reject H0
No significant difference found.


In [17]:
category_profit = group_comparison_test(
    df,
    "Category",
    "Profit"
)

category_sales = group_comparison_test(
    df,
    "Category",
    "Total Sales"
)


Category vs Profit

Furniture: Normality p-value = 0.000000e+00 : Not Normal
Office Supplies: Normality p-value = 0.000000e+00 : Not Normal
Technology: Normality p-value = 0.000000e+00 : Not Normal

Levene test p-value: 0.000000e+00

Selected test: Kruskal-Wallis
Statistic: 1752.7204
P-value: 0.000000e+00
Reject H0
Significant difference exists.

Category vs Total Sales

Furniture: Normality p-value = 0.000000e+00 : Not Normal
Office Supplies: Normality p-value = 0.000000e+00 : Not Normal
Technology: Normality p-value = 0.000000e+00 : Not Normal

Levene test p-value: 0.000000e+00

Selected test: Kruskal-Wallis
Statistic: 8028.1868
P-value: 0.000000e+00
Reject H0
Significant difference exists.


In [18]:
category_returned = chi_square_test(
    df,
    "Category",
    "Returned"
)


Category vs Returned


Contingency Table:
Returned             0     1
Category                    
Furniture         8967   628
Office Supplies  28546  1741
Technology        9177   611

Chi-Square Statistic: 9.4753
Degrees of Freedom: 2
P-value: 8.759406e-03
Reject H0
Result: Significant association exists.


In [22]:
region_returned = chi_square_test(
    df,
    "Region",
    "Returned"
)


Region vs Returned


Contingency Table:
Returned           0    1
Region                   
Africa          4453    0
Canada           376    0
Caribbean       1616   62
Central         9701  548
Central Asia    1973   75
EMEA            4835    0
East            2681  151
North           3972  596
North Asia      1928  410
Oceania         3339  148
South           6143  347
Southeast Asia  2976  153
West            2697  490

Chi-Square Statistic: 2144.0270
Degrees of Freedom: 12
P-value: 0.000000e+00
Reject H0
Result: Significant association exists.


In [23]:
results = [
    discount_profit,
    discount_sales,
    shipping_profit,
    shipping_sales,
    quantity_profit,
    quantity_sales,
    sales_profit,
    market_profit,
    market_sales,
    region_profit,
    region_sales,
    segment_profit,
    segment_sales,
    category_profit,
    category_sales,
    category_returned,
    region_returned
]

results_df = pd.DataFrame(results)

results_df

,Variable 1,Variable 2,Test,Statistic,P-value,Result,Grouping Variable,Value Variable
0,Discount,Profit,Spearman Correlation,-0.598178,0.000000e+00,Significant relationship exists.,NaN,NaN
1,Discount,Total Sales,Spearman Correlation,-0.200367,0.000000e+00,Significant relationship exists.,NaN,NaN
2,Shipping Cost,Profit,Spearman Correlation,0.448015,0.000000e+00,Significant relationship exists.,NaN,NaN
3,Shipping Cost,Total Sales,Spearman Correlation,0.853137,0.000000e+00,Significant relationship exists.,NaN,NaN
4,Quantity,Profit,Spearman Correlation,0.198589,0.000000e+00,Significant relationship exists.,NaN,NaN
5,Quantity,Total Sales,Spearman Correlation,0.664129,0.000000e+00,Significant relationship exists.,NaN,NaN
6,Total Sales,Profit,Spearman Correlation,0.544751,0.000000e+00,Significant relationship exists.,NaN,NaN
7,NaN,NaN,Kruskal-Wallis,421.583021,6.385676e-88,NaN,Market,Profit
8,NaN,NaN,Kruskal-Wallis,3851.094272,0.000000e+00,NaN,Market,Total Sales
9,NaN,NaN,Kruskal-Wallis,1740.429195,0.000000e+00,NaN,Region,Profit


In [24]:
significant_results = results_df[
    results_df["P-value"] < 0.05
]

significant_results

,Variable 1,Variable 2,Test,Statistic,P-value,Result,Grouping Variable,Value Variable
0,Discount,Profit,Spearman Correlation,-0.598178,0.000000e+00,Significant relationship exists.,NaN,NaN
1,Discount,Total Sales,Spearman Correlation,-0.200367,0.000000e+00,Significant relationship exists.,NaN,NaN
2,Shipping Cost,Profit,Spearman Correlation,0.448015,0.000000e+00,Significant relationship exists.,NaN,NaN
3,Shipping Cost,Total Sales,Spearman Correlation,0.853137,0.000000e+00,Significant relationship exists.,NaN,NaN
4,Quantity,Profit,Spearman Correlation,0.198589,0.000000e+00,Significant relationship exists.,NaN,NaN
5,Quantity,Total Sales,Spearman Correlation,0.664129,0.000000e+00,Significant relationship exists.,NaN,NaN
6,Total Sales,Profit,Spearman Correlation,0.544751,0.000000e+00,Significant relationship exists.,NaN,NaN
7,NaN,NaN,Kruskal-Wallis,421.583021,6.385676e-88,NaN,Market,Profit
8,NaN,NaN,Kruskal-Wallis,3851.094272,0.000000e+00,NaN,Market,Total Sales
9,NaN,NaN,Kruskal-Wallis,1740.429195,0.000000e+00,NaN,Region,Profit


In [25]:
results_df["Conclusion"] = results_df["P-value"].apply(
    lambda p: "Significant" if p < 0.05 else "Not Significant"
)

results_df

,Variable 1,Variable 2,Test,Statistic,P-value,Result,Grouping Variable,Value Variable,Conclusion
0,Discount,Profit,Spearman Correlation,-0.598178,0.000000e+00,Significant relationship exists.,NaN,NaN,Significant
1,Discount,Total Sales,Spearman Correlation,-0.200367,0.000000e+00,Significant relationship exists.,NaN,NaN,Significant
2,Shipping Cost,Profit,Spearman Correlation,0.448015,0.000000e+00,Significant relationship exists.,NaN,NaN,Significant
3,Shipping Cost,Total Sales,Spearman Correlation,0.853137,0.000000e+00,Significant relationship exists.,NaN,NaN,Significant
4,Quantity,Profit,Spearman Correlation,0.198589,0.000000e+00,Significant relationship exists.,NaN,NaN,Significant
5,Quantity,Total Sales,Spearman Correlation,0.664129,0.000000e+00,Significant relationship exists.,NaN,NaN,Significant
6,Total Sales,Profit,Spearman Correlation,0.544751,0.000000e+00,Significant relationship exists.,NaN,NaN,Significant
7,NaN,NaN,Kruskal-Wallis,421.583021,6.385676e-88,NaN,Market,Profit,Significant
8,NaN,NaN,Kruskal-Wallis,3851.094272,0.000000e+00,NaN,Market,Total Sales,Significant
9,NaN,NaN,Kruskal-Wallis,1740.429195,0.000000e+00,NaN,Region,Profit,Significant


# Statistical Results Summary

## Correlation Results

| Relationship                | Test     | Correlation | Conclusion                     |
| --------------------------- | -------- | ----------: | ------------------------------ |
| Discount vs Profit           | Spearman |      -0.598 | Significant, moderate negative |
| Discount vs Total Sales      | Spearman |      -0.200 | Significant, weak negative     |
| Shipping Cost vs Profit      | Spearman |       0.448 | Significant, moderate positive |
| Shipping Cost vs Total Sales | Spearman |       0.853 | Significant, strong positive   |
| Quantity vs Profit           | Spearman |       0.199 | Significant, weak positive     |
| Quantity vs Total Sales      | Spearman |       0.664 | Significant, moderate positive |
| Total Sales vs Profit        | Spearman |       0.545 | Significant, moderate positive |

## Group Comparison Results

| Group    | Profit                    | Total Sales               |
| -------- | ------------------------- | ------------------------- |
| Market   | Significant difference    | Significant difference    |
| Region   | Significant difference    | Significant difference    |
| Segment  | No significant difference | No significant difference |
| Category | Significant difference    | Significant difference    |

All significant group differences were identified using the **Kruskal-Wallis test**.

## Category & Returned

**Chi-Square test:** Category and Returned have a significant association
**p = 0.0088**

## Important Final Points

* **Discount** has a moderate negative relationship with Profit and a weak negative relationship with Total Sales.
* **Shipping Cost** has a moderate positive relationship with Profit and a strong positive relationship with Total Sales.
* **Quantity** has a weak positive relationship with Profit and a moderate positive relationship with Total Sales.
* **Total Sales** has a moderate positive relationship with Profit.
* **Market, Region, and Category** show significant differences in both Profit and Total Sales.
* **Segment** does not show a significant difference in either Profit or Total Sales.
* **Category and Returned** have a statistically significant association.

> **Note:** Statistical significance indicates that a relationship or difference is unlikely to be due to random variation; it does not by itself imply causation.
